# How far are KV-cache eviction policies from the offline optimum?
**Design & Analysis of Algorithms project** — model: Qwen2.5-0.5B

**Problem.** While generating text, an LLM stores keys/values (the *KV cache*) of all previous tokens. With limited memory we can keep only `k` of them, so at every step an **eviction policy** must decide which token to throw away. Eviction is **permanent**: once a token's KV is dropped it is never recomputed.

**This is an online problem** (decisions are made without knowing the future), just like page replacement. So we can ask the classic algorithms question: *how close are practical online policies to the best possible offline (future-knowing) policy?*

**What this notebook does**
1. Extracts the full attention matrix `A[t, j]` (how much token *t* looks at token *j*).
2. Implements 6 online policies (Random, Sliding Window, StreamingLLM, Heavy-Hitter-only, H2O, Hybrid).
3. Computes the **exact offline optimum** by a reduction to **min-cost flow**, plus a Belady-style greedy oracle.
4. Judges policies three ways: attention retained (competitive ratio), real perplexity with a pruned cache, and a needle-in-a-haystack recall test.

Runtime: ~25–40 min on a free Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU).

## 0. Install (pinned versions)

In [ ]:
!pip install -q "transformers==4.46.3" "datasets==3.1.0" "ortools==9.12.4544"
# If you see import errors later, use Runtime → Restart session and run from the next cell.

## 1. Core algorithms (no model needed)
Written to a file so the notebook stays readable. Key pieces:
- `Policy` subclasses = online algorithms; `select_evictions` may only use information from earlier steps.
- `optimal_schedule` = **exact offline optimum** via min-cost flow (read the docstring — this is the main DAA contribution).
- `simulate` = runs any policy on an attention matrix and returns attention retained per step.

In [ ]:
# Core algorithms: policies, simulator, and exact offline optimum
# Source: src/core.py (no longer written inline)
import sys, os
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if p not in sys.path: sys.path.insert(0, p)  # so src/ is importable from repo root or notebooks/
from src.core import *


## 1b. Imitation scorer (learned policy)
**Idea in one sentence:** the exact optimum (min-cost flow) tells us *which tokens were worth keeping*; we train a tiny logistic regression to predict that from information available **online**, then use it as an eviction policy.

- **Teacher** = optimal schedule `end[j]` (needs the future). **Student** = logistic regression on past-only features: age, average / last-step / EMA attention, popularity vs the rest of the cache, is-sink, budget.
- **Label** = 1 if the optimum still holds token *j* `k/2` steps from now. Training states come from rollouts of the optimum, H2O, Hybrid and SlidingWindow, so the student also sees states created by imperfect policies.
- **Online rule** = evict the cached token with the lowest predicted keep-probability. O(k) per step, like H2O.
- **No leakage:** a model is never evaluated on the sequences or text type it was trained on (see section 10b).
- **DAgger** (`dagger_train`, used in section 10d): the student is also trained on the states *it* visits, relabelled by the optimum.

In [ ]:
# Imitation scorer: learn to copy the offline optimum
# Source: src/imitation.py
from src.imitation import *


## 2. Model side
- `attention_matrix`: one forward pass with full cache → `A` (averaged over 24 layers × 14 heads) and the full-cache loss per token.
- `run_with_eviction`: decodes token by token while **physically deleting** evicted entries from the KV cache. Kept tokens keep their original RoPE positions. (Verified during development: with no eviction it matches the full forward pass to 1e-6, and pruning matches an attention-mask reference implementation.)

In [ ]:
# Model side: attention extraction and pruned-cache decoding
# Source: src/model.py
from src.model import *


## 3. Configuration

In [ ]:
MODEL_NAME   = "Qwen/Qwen2.5-0.5B"
T            = 512                  # tokens per sequence
BUDGETS      = [0.1, 0.2, 0.3, 0.5] # cache size k as a fraction of T (attention experiment)
N_SEQ        = 3                    # sequences per text type (attention experiment)
PPL_BUDGETS  = [0.1, 0.25]          # real-decoding perplexity experiment (slower)
NEEDLE_DEPTHS = [0.1, 0.3, 0.5, 0.7, 0.9]
NEEDLE_BUDGET = 0.2
SEED = 0

import os
os.environ["USE_TF"] = "0"; os.environ["USE_TORCH"] = "1"   # stop transformers from importing TensorFlow
import time, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src.core import *
from src.model import *
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("results", exist_ok=True)
POLICY_ORDER = ["Random", "SlidingWindow", "StreamingLLM", "HeavyHitter-only", "H2O", "Hybrid",
                "Belady-Greedy (offline)", "Optimal (offline)"]
COLORS = dict(zip(POLICY_ORDER, plt.cm.tab10.colors))

## 4. Load the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, attn_implementation="eager").to(device).eval()
print("device:", device, "| layers:", model.config.num_hidden_layers, "| heads:", model.config.num_attention_heads)

## 5. Data: three kinds of text
Different text has different "memory patterns", so the best policy may differ:
- **Wikipedia** (WikiText-2): long-range references to topics and names.
- **Code** (Python standard-library source, available offline): variables defined far above are reused.
- **Dialogue** (UltraChat): turn-based, mostly local context.

In [ ]:
def load_texts():
    import inspect, argparse, collections, http.client
    from datasets import load_dataset
    wiki = "".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"])
    code_text = "\n\n".join(inspect.getsource(m) for m in [argparse, collections, http.client])
    try:
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft", streaming=True)
        convs = []
        for i, ex in enumerate(ds):
            convs.append("\n".join(f"{m['role'].capitalize()}: {m['content']}" for m in ex["messages"]))
            if i >= 80: break
        dialog = "\n\n".join(convs)
    except Exception as e:
        print("Dialogue dataset failed, skipping it:", e); dialog = None
    texts = {"wikipedia": wiki, "code": code_text}
    if dialog: texts["dialogue"] = dialog
    return texts

TEXTS = load_texts()
SEQS = {}
for name, text in TEXTS.items():
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    starts = np.linspace(0, len(ids) - T - 1, N_SEQ).astype(int)
    SEQS[name] = [ids[s:s + T].clone() for s in starts]
    print(f"{name:10s}: {len(ids):7d} tokens -> {N_SEQ} sequences of {T}")

## 6. Look at the attention first
Bright diagonal = attention to recent tokens. Bright first columns = **attention sinks** (the model dumps attention on the first tokens). Bright vertical stripes = **heavy hitters** (tokens re-read many times). These three patterns are exactly what the policies bet on.

In [ ]:
A_STORE = {}
for name, seqs in SEQS.items():
    for si, ids in enumerate(seqs):
        A_STORE[(name, si)] = attention_matrix(model, ids)   # (A, full-cache nll)

fig, axes = plt.subplots(1, len(SEQS), figsize=(5 * len(SEQS), 4.5))
for ax, name in zip(np.atleast_1d(axes), SEQS):
    A = A_STORE[(name, 0)][0]
    ax.imshow(np.log10(A + 1e-6), cmap="magma", vmin=-4, vmax=0)
    ax.set_title(f"{name}: log10 attention"); ax.set_xlabel("key token j"); ax.set_ylabel("query step t")
    print(f"{name:10s} share of attention on first 4 tokens: {A[64:, :4].sum(1).mean():.2f}")
plt.tight_layout(); plt.savefig("results/attention_maps.png", dpi=150); plt.show()

## 7. Experiment 1 — attention retained & competitive ratio
For each sequence and budget we simulate every policy and the exact optimum.
**Score** = average attention mass that lands on cached tokens (over steps where eviction is active).
**Competitive ratio** = policy score ÷ optimal score (1.0 = as good as knowing the future).

In [ ]:
rows, OPT_SECS = [], []
for (name, si), (A, _) in A_STORE.items():
    for b in BUDGETS:
        k = int(b * T)
        pols, secs = all_policies(A, k); OPT_SECS.append(secs)
        for p in pols:
            mass, dt = simulate(A, k, p)
            rows.append(dict(text=name, seq=si, budget=b, k=k, policy=p.name, online=p.online,
                             mass=mass[k:].mean(), us_per_step=dt * 1e6))
df = pd.DataFrame(rows)
opt = df[df.policy == "Optimal (offline)"].set_index(["text", "seq", "budget"])["mass"]
df["ratio"] = [m / opt[(t, s, b)] for t, s, b, m in zip(df.text, df.seq, df.budget, df.mass)]
df.to_csv("results/exp1_attention.csv", index=False)
print(f"Optimal schedule (min-cost flow, {T*(T+1)//2:,} arcs): {np.mean(OPT_SECS):.2f}s per instance")
summary1 = df.groupby(["text", "budget", "policy"])["ratio"].mean().unstack("policy")[POLICY_ORDER]
summary1.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(SEQS), figsize=(5.5 * len(SEQS), 4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), SEQS):
    s = df[df.text == name].groupby(["budget", "policy"])["ratio"].mean().unstack()
    for p in POLICY_ORDER:
        ax.plot(s.index, s[p], "o-", label=p, color=COLORS[p], ls="--" if "offline" in p else "-")
    ax.set_title(name); ax.set_xlabel("cache budget (fraction of tokens)"); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel("competitive ratio (vs optimal)")
np.atleast_1d(axes)[-1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig("results/exp1_competitive_ratio.png", dpi=150); plt.show()

### Which tokens does each policy keep?
Each image shows cache contents over time (black = token *j* is in the cache at step *t*). Sliding window is a band; StreamingLLM adds sink columns; H2O keeps scattered heavy hitters; the optimum shows what the *ideal* pattern looks like.

In [ ]:
def occupancy(A, k, policy):
    T_ = A.shape[0]; policy.reset(T_, k); cache = []; M = np.zeros((T_, T_))
    for t in range(T_):
        cache.append(t); ev = set(policy.select_evictions(t, cache))
        cache = [j for j in cache if j not in ev]; M[t, cache] = 1
        r = A[t, cache]; policy.observe(t, cache, r / r.sum())
    return M

name0 = list(SEQS)[0]; A0 = A_STORE[(name0, 0)][0]; k0 = int(0.2 * T)
show = [SlidingWindow(), StreamingLLM(), HeavyHitter(0.5, "H2O"), ScheduledPolicy(optimal_schedule(A0, k0)[0])]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, p in zip(axes, show):
    ax.imshow(occupancy(A0, k0, p), cmap="Greys", interpolation="nearest")
    ax.set_title(p.name); ax.set_xlabel("token j"); ax.set_ylabel("step t")
plt.suptitle(f"Cache contents over time ({name0}, k = {k0})"); plt.tight_layout()
plt.savefig("results/cache_occupancy.png", dpi=150); plt.show()

## 8. Experiment 2 — real perplexity with a pruned cache
The attention score is only a *proxy*. Here we actually delete KV entries and measure **perplexity** (how surprised the model is by the true next token; lower is better). ΔPPL% = increase over the full cache, on the same tokens. Offline policies replay their schedule computed from full-cache attention.

In [ ]:
rows2 = []
for name in SEQS:
    ids = SEQS[name][0]; A, nll_full = A_STORE[(name, 0)]
    for b in PPL_BUDGETS:
        k = int(b * T); pols, _ = all_policies(A, k)
        ppl_full = np.exp(nll_full[k:].mean())
        for p in pols:
            t0 = time.time()
            nll, _ = run_with_eviction(model, ids, k, p)
            mass, _ = simulate(A, k, p)
            ppl = np.exp(nll[k:].mean())
            rows2.append(dict(text=name, budget=b, policy=p.name, ppl=ppl, ppl_full=ppl_full,
                              dppl_pct=100 * (ppl / ppl_full - 1), mass=mass[k:].mean()))
            print(f"{name:10s} b={b:.2f} {p.name:26s} PPL {ppl:8.2f} (full {ppl_full:.2f})  [{time.time()-t0:.0f}s]")
df2 = pd.DataFrame(rows2); df2.to_csv("results/exp2_perplexity.csv", index=False)
df2.pivot_table(index=["text", "budget"], columns="policy", values="dppl_pct")[POLICY_ORDER].round(1)

In [ ]:
from scipy.stats import spearmanr
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
piv = df2.groupby("policy")["dppl_pct"].mean().reindex(POLICY_ORDER)
axes[0].bar(range(len(piv)), piv.values, color=[COLORS[p] for p in piv.index])
axes[0].set_xticks(range(len(piv))); axes[0].set_xticklabels(piv.index, rotation=35, ha="right")
axes[0].set_ylabel("perplexity increase vs full cache (%)"); axes[0].set_title("Real quality loss (lower = better)")
axes[0].set_yscale("symlog")
for p in POLICY_ORDER:
    d = df2[df2.policy == p]; axes[1].scatter(d.mass, d.dppl_pct, color=COLORS[p], label=p)
rho, pval = spearmanr(df2.mass, df2.dppl_pct)
axes[1].set_xlabel("attention retained (proxy)"); axes[1].set_ylabel("ΔPPL %"); axes[1].set_yscale("symlog")
axes[1].set_title(f"Is attention a good proxy?  Spearman ρ = {rho:.2f} (p={pval:.1e})")
axes[1].legend(fontsize=7); plt.tight_layout(); plt.savefig("results/exp2_perplexity.png", dpi=150); plt.show()

## 9. Experiment 3 — needle in a haystack
We hide "The secret code is XXXX" at different depths inside ~500 tokens of Wikipedia text, then ask for it at the end. A policy **passes** if the model still answers correctly with its pruned cache. (During the few answer tokens the cache may grow; no extra eviction.)

In [ ]:
class FullCache(Policy):
    name = "Full cache"
    def select_evictions(self, t, cache): return []

filler = tok(TEXTS["wikipedia"][200000:260000], add_special_tokens=False).input_ids
q_ids = tok.encode("\nQuestion: What is the secret code? Answer: The secret code is", add_special_tokens=False)
rng = np.random.default_rng(SEED); rows3 = []
k = int(NEEDLE_BUDGET * T)
for depth in NEEDLE_DEPTHS:
    code_str = str(rng.integers(1000, 9999))
    needle = tok.encode(f" The secret code is {code_str}. ", add_special_tokens=False)
    n_fill = T - len(needle) - len(q_ids); pos = int(depth * n_fill)
    ids = torch.tensor(filler[:pos] + needle + filler[pos:n_fill] + q_ids)
    A, _ = attention_matrix(model, ids)
    pols, _ = all_policies(A, k)
    for p in [FullCache()] + pols:
        _, gen = run_with_eviction(model, ids, T if p.name == "Full cache" else k, p, gen_tokens=6)
        ans = tok.decode(gen)
        rows3.append(dict(depth=depth, policy=p.name, answer=ans.strip(), success=code_str in ans.replace(" ", "")))
    print(f"depth {depth}: code {code_str} | " + ", ".join(f"{r['policy'].split(' ')[0]}={'✓' if r['success'] else '✗'}" for r in rows3[-len(pols)-1:]))
df3 = pd.DataFrame(rows3); df3.to_csv("results/exp3_needle.csv", index=False)
acc = df3.groupby("policy")["success"].mean().reindex(["Full cache"] + POLICY_ORDER)
ax = acc.plot.bar(figsize=(9, 3.5), color=["grey"] + [COLORS[p] for p in POLICY_ORDER])
ax.set_ylabel("retrieval accuracy"); ax.set_title(f"Needle retrieval at k = {k} ({NEEDLE_BUDGET:.0%} of tokens)")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.savefig("results/exp3_needle.png", dpi=150); plt.show()

## 10. Overhead — cost of each decision

In [ ]:
ov = df.groupby("policy")["us_per_step"].mean().reindex(POLICY_ORDER).round(1).to_frame("µs per step (python)")
ov["theory"] = ["O(1)", "O(1)", "O(1)", "O(k)", "O(k)", "O(k)", "O(k) + O(T²) precompute", "min-cost flow on O(T²) arcs"]
ov

## 10b. Extension — imitation scorer vs. the optimum
Two honest ways to train, both leakage-free:
- **Imitation (cross-domain):** for each text type, train only on the *other* text types (e.g. test on code, train on Wikipedia + dialogue).
- **Imitation (in-domain):** for each sequence, train on the *other sequences of the same text type* (leave-one-sequence-out).

Tests: **A** competitive ratio vs budget · **D** keep-set overlap (Jaccard) with the optimum · **B** real perplexity · **C** needle retrieval · µs per step.

In [ ]:
from src.imitation import *
from collections import defaultdict

# ---- exact optimum for every (text, sequence, budget): the teacher ----
OPT_END = {}
for (name, si), (A, _) in A_STORE.items():
    for b in BUDGETS:
        OPT_END[(name, si, b)] = optimal_schedule(A, int(b * T))[0]

def items_for(keys):
    return [(A_STORE[(n, s)][0], int(b * T), OPT_END[(n, s, b)]) for (n, s) in keys for b in BUDGETS]

# ---- train the students (never on the data they are tested on) ----
IMIT_X, IMIT_IN = {}, {}
for name in SEQS:
    other = [(n, s) for (n, s) in A_STORE if n != name]
    IMIT_X[name] = train_imitation(items_for(other), name="Imitation (cross-domain)")
    print(f"cross-domain  test={name:10s} trained on {len(other)} sequences, {IMIT_X[name].n_train:,} rows")
for (name, si) in A_STORE:
    same = [(n, s) for (n, s) in A_STORE if n == name and s != si]
    IMIT_IN[(name, si)] = train_imitation(items_for(same), name="Imitation (in-domain)")
print("done")

In [ ]:
BASE_NAMES = ["StreamingLLM", "SlidingWindow", "H2O", "Hybrid"]
def fresh_baselines():
    return [StreamingLLM(), SlidingWindow(), HeavyHitter(0.5, "H2O"), Hybrid()]

rows4 = []
for (name, si), (A, _) in A_STORE.items():
    for b in BUDGETS:
        k = int(b * T); end = OPT_END[(name, si, b)]
        opt_pol = ScheduledPolicy(end)
        opt_mass = simulate(A, k, opt_pol)[0][k:].mean()
        M_opt = occupancy(A, k, ScheduledPolicy(end))
        pols = [IMIT_X[name], IMIT_IN[(name, si)]] + fresh_baselines()
        for p in pols:
            mass, dt = simulate(A, k, p)
            M = occupancy(A, k, p)[k:]; Mo = M_opt[k:]
            jac = ((M * Mo).sum(1) / np.maximum(((M + Mo) > 0).sum(1), 1)).mean()   # |A∩B| / |A∪B| per step
            rows4.append(dict(text=name, seq=si, budget=b, policy=p.name,
                              ratio=mass[k:].mean() / opt_mass, jaccard=jac, us_per_step=dt * 1e6))
df4 = pd.DataFrame(rows4); df4.to_csv("results/imitation_exp1.csv", index=False)

ORDER4 = ["Imitation (cross-domain)", "Imitation (in-domain)", "H2O", "Hybrid", "StreamingLLM", "SlidingWindow"]
print("Mean competitive ratio (A) | Jaccard overlap with the optimum's cache (D) | µs per decision")
tab = df4.groupby("policy")[["ratio", "jaccard", "us_per_step"]].mean().reindex(ORDER4).round(3)
display(tab)
display(df4.groupby(["text", "policy"])["ratio"].mean().unstack()[ORDER4].round(3))

fig, axes = plt.subplots(1, len(SEQS), figsize=(5.5 * len(SEQS), 4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), SEQS):
    s = df4[df4.text == name].groupby(["budget", "policy"])["ratio"].mean().unstack()
    for p in ORDER4:
        ax.plot(s.index, s[p], "o", label=p, lw=2.5 if p.startswith("Imitation") else 1.2,
                ls="-" if p.startswith("Imitation") else "--")
    ax.set_title(name); ax.set_xlabel("cache budget (fraction of tokens)"); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel("competitive ratio (vs optimal)")
np.atleast_1d(axes)[-1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig("results/imitation_ratio.png", dpi=150); plt.show()

# what did the student learn from the optimum?  (standardised weights, mean over cross-domain models)
W = np.mean([m.w for m in IMIT_X.values()], axis=0)
plt.figure(figsize=(7, 3.5)); plt.barh(FEATURES, W, color=["tab:green" if w > 0 else "tab:red" for w in W])
plt.axvline(0, color="k", lw=.8); plt.xlabel("weight  (+ = keep, − = evict)")
plt.title("What the optimum rewards (learned weights)"); plt.tight_layout()
plt.savefig("results/imitation_weights.png", dpi=150); plt.show()

In [ ]:
# Test B: real perplexity with a physically pruned cache (same protocol as section 8)
rows5 = []
for name in SEQS:
    ids = SEQS[name][0]; A, nll_full = A_STORE[(name, 0)]
    for b in PPL_BUDGETS:                        # note: 0.25 was never a training budget -> tests generalisation
        k = int(b * T); ppl_full = np.exp(nll_full[k:].mean())
        p = IMIT_X[name]
        nll, _ = run_with_eviction(model, ids, k, p)
        ppl = np.exp(nll[k:].mean()); mass, _ = simulate(A, k, p)
        rows5.append(dict(text=name, budget=b, policy=p.name, ppl=ppl, dppl_pct=100 * (ppl / ppl_full - 1), mass=mass[k:].mean()))
        print(f"{name:10s} b={b:.2f} Imitation PPL {ppl:8.2f} (full {ppl_full:.2f})")
df5 = pd.DataFrame(rows5); df5.to_csv("results/imitation_exp2_ppl.csv", index=False)
cmp = pd.concat([df2[df2.policy.isin(["StreamingLLM", "H2O", "Hybrid", "Optimal (offline)"])], df5])
display(cmp.pivot_table(index=["text", "budget"], columns="policy", values="dppl_pct").round(1))

In [ ]:
# Test C: needle in a haystack (same needles as section 9; requires that cell to have run)
rng = np.random.default_rng(SEED); rows6 = []
k = int(NEEDLE_BUDGET * T); pol = IMIT_X["wikipedia"]          # filler is Wikipedia -> trained on code + dialogue only
for depth in NEEDLE_DEPTHS:
    code_str = str(rng.integers(1000, 9999))
    needle = tok.encode(f" The secret code is {code_str}. ", add_special_tokens=False)
    n_fill = T - len(needle) - len(q_ids); pos = int(depth * n_fill)
    ids = torch.tensor(filler[:pos] + needle + filler[pos:n_fill] + q_ids)
    _, gen = run_with_eviction(model, ids, k, pol, gen_tokens=6)
    ans = tok.decode(gen)
    rows6.append(dict(depth=depth, policy=pol.name, answer=ans.strip(), success=code_str in ans.replace(" ", "")))
    print(f"depth {depth}: {'✓' if rows6[-1]['success'] else '✗'}  answer={ans.strip()!r}  (true {code_str})")
df6 = pd.DataFrame(rows6); df6.to_csv("results/imitation_exp3_needle.csv", index=False)
acc2 = pd.concat([acc, pd.Series({pol.name: df6.success.mean()})])
print("\nNeedle accuracy:"); print(acc2.round(2).to_string())

In [ ]:
print("=== Imitation scorer: numbers to quote ===")
print(tab.to_string())
H2 = df4[df4.policy == "H2O"]["ratio"].mean(); I1 = df4[df4.policy == "Imitation (cross-domain)"]["ratio"].mean()
print(f"\nCross-domain imitation vs H2O: {I1:.3f} vs {H2:.3f}  ->  " + ("beats H2O" if I1 > H2 else "does NOT beat H2O (report this honestly)"))
best_base = df4[df4.policy.isin(BASE_NAMES)].groupby("policy")["ratio"].mean().idxmax()
print(f"Best hand-designed online baseline here: {best_base}")
gap = df4[df4.policy == "Imitation (in-domain)"]["ratio"].mean() - I1
print(f"In-domain minus cross-domain ratio: {gap:+.3f}  (large gap = the learned rule is text-type specific)")
print("Weights (standardised):", {f: round(float(w), 2) for f, w in zip(FEATURES, W)})

### How to explain the imitation scorer (viva)
- **What:** a logistic regression that predicts, from past-only features, whether the offline optimum would still hold a token later; evict the lowest score.
- **Why it is not cheating:** the optimum is used only to make training labels. At test time the policy sees only the past. It is never evaluated on text it was trained on.
- **Why logistic regression:** it is tiny and O(k) per step like H2O, and the weights are readable (see the weight plot), so you can say what the optimum rewards.
- **Weaknesses to state:** labels come from the attention-proxy optimum, not from perplexity; the student is trained on rollouts of a few policies, so it can still meet unfamiliar states (distribution shift); with only 3 text types, cross-domain results are noisy.

## 10c. Scaled, harder tests for the imitation scorer
The first version used 3 sequences per text type. Here we use up to **12 non-overlapping sequences per type** and **7 budgets** (3 of them never seen in training), then test in five different ways:

| # | Test | The question it answers |
|---|------|--------------------------|
| 1 | **Scoreboard with error bars** | Is the learned policy better than hand-designed ones, with statistics (bootstrap CI, win-rate, paired Wilcoxon)? Does it hold on unseen cache sizes? |
| 2 | **Where does the gap come from?** | The attention a policy loses vs the optimum, split into *sink*, *recent* and *long-range* tokens. |
| 3 | **Handed the optimum's memory** | If we give each policy the optimum's exact cache, does it drop the *same token* the optimum drops? |
| 4 | **Stress traces** | What happens on attention patterns it never saw: pure recency, sinks, periodic re-reads, a sudden topic change, an unpredictable late burst? |
| 5 | **Real quality at scale** | Perplexity and needle retrieval over many sequences, with CIs, and whether attention-retained really predicts quality. |

Set `PRESET = "quick"` in the next cell for a fast sanity run first. The training students never see the text they are tested on: cross-domain = trained on the other text types, in-domain = 4-fold cross-validation over sequences.

In [ ]:
PRESET = "full"      # "quick" = about a quarter of the work, good for a first sanity run
CFG = {"quick": dict(n_seq=4, n_ppl=1, n_reps=1),
       "full":  dict(n_seq=12, n_ppl=4, n_reps=3)}[PRESET]
TRAIN_BUDGETS = [0.1, 0.2, 0.3, 0.5]
TEST_BUDGETS  = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
UNSEEN = [b for b in TEST_BUDGETS if b not in TRAIN_BUDGETS]      # 0.05, 0.15, 0.4: never used for training

# non-overlapping T-token chunks spread evenly through each text
BIG = {}
for name, text in TEXTS.items():
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    n_avail = len(ids) // T
    pick = np.unique(np.linspace(0, n_avail - 1, min(CFG["n_seq"], n_avail)).round().astype(int))
    BIG[name] = [ids[p * T:(p + 1) * T].clone() for p in pick]
    print(f"{name:10s}: {len(BIG[name])} non-overlapping sequences")
A_BIG = {}
for name, seqs in BIG.items():
    for si, ids in enumerate(seqs):
        A_BIG[(name, si)] = attention_matrix(model, ids)

In [ ]:
# exact optimum for every (sequence, budget). Small budgets are slowest (about 1-3 s each).
t0 = time.time(); OPT_BIG = {}
for (n, s), (A, _) in A_BIG.items():
    for b in TEST_BUDGETS:
        OPT_BIG[(n, s, b)] = optimal_schedule(A, int(b * T))[0]
print(f"{len(OPT_BIG)} optimal schedules in {time.time() - t0:.0f}s")

def big_items(keys, budgets=TRAIN_BUDGETS):
    return [(A_BIG[key][0], int(b * T), OPT_BIG[(*key, b)]) for key in keys for b in budgets]

# cross-domain students: never see the test text type
BIG_X = {}
for name in BIG:
    keys = [(n, s) for n in BIG if n != name for s in range(len(BIG[n]))]
    BIG_X[name] = train_imitation(big_items(keys), name="Imitation (cross-domain)")
# in-domain students: 4-fold cross-validation over sequences of the same text type
BIG_IN = {}
for name in BIG:
    n = len(BIG[name]); nf = min(4, n); fold = np.arange(n) % nf
    for f in range(nf):
        m = train_imitation(big_items([(name, s) for s in range(n) if fold[s] != f]), name="Imitation (in-domain)")
        for s in range(n):
            if fold[s] == f: BIG_IN[(name, s)] = m
print("students trained")

In [ ]:
POL_ORDER = ["Imitation (cross-domain)", "Imitation (in-domain)", "H2O", "Hybrid", "StreamingLLM", "SlidingWindow"]

def teacher_forced(A, k, end, policy):
    """Hand the policy the OPTIMUM's cache at every step and ask which token it would drop.
    Returns (fraction of steps where it drops the optimum's token, chance level)."""
    T_ = A.shape[0]; policy.reset(T_, k); cache = []; hit = n = 0; chance = 0.0
    for t in range(T_):
        cache.append(t)
        opt_ev = {j for j in cache[:-1] if end[j] < t}
        if len(cache) > k and opt_ev:
            ev = policy.select_evictions(t, cache)
            hit += bool(ev) and ev[0] in opt_ev; n += 1; chance += len(opt_ev) / (len(cache) - 1)
        cache = [j for j in cache if j not in opt_ev]
        row = A[t, cache]; policy.observe(t, cache, row / max(row.sum(), 1e-12))
    return hit / max(n, 1), chance / max(n, 1)

rows = []; t0 = time.time()
Tm, Jm = np.indices((T, T))
for (name, si), (A, _) in A_BIG.items():
    for b in TEST_BUDGETS:
        k = int(b * T); end = OPT_BIG[(name, si, b)]
        Mo = occupancy(A, k, ScheduledPolicy(end))
        opt_mass = (A * Mo)[k:].sum(1).mean()
        SINK = (Jm < 4); REC = ((Tm - Jm) < k / 4) & ~SINK
        for p in [BIG_X[name], BIG_IN[(name, si)], HeavyHitter(0.5, "H2O"), Hybrid(), StreamingLLM(), SlidingWindow()]:
            M = occupancy(A, k, p)
            d = (A * (Mo - M))[k:]                              # attention the optimum keeps minus what the policy keeps
            mid = ~(SINK | REC)[k:]
            agree, chance = teacher_forced(A, k, end, p)
            rows.append(dict(text=name, seq=si, budget=b, policy=p.name,
                             ratio=(A * M)[k:].sum(1).mean() / opt_mass,
                             lost_sink=(d * SINK[k:]).sum(1).mean() / opt_mass,
                             lost_recent=(d * REC[k:]).sum(1).mean() / opt_mass,
                             lost_longrange=(d * mid).sum(1).mean() / opt_mass,
                             agree=agree, chance=chance))
big = pd.DataFrame(rows); big["unseen"] = big.budget.isin(UNSEEN)
big.to_csv("results/big_exp.csv", index=False)
print(f"{len(big)} runs in {time.time() - t0:.0f}s")

In [ ]:
# ---------------- Test 1: scoreboard with error bars ----------------
from scipy import stats
def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); r = np.random.default_rng(seed)
    return np.percentile(r.choice(x, (n, len(x))).mean(1), [2.5, 97.5])

per_seq = big.groupby(["text", "seq", "policy"])["ratio"].mean().unstack()[POL_ORDER]   # one number per sequence
def fmt(x): lo, hi = boot_ci(x); return f"{np.mean(x):.3f} [{lo:.3f}, {hi:.3f}]"
print("Competitive ratio, mean over budgets, [95% bootstrap CI over sequences]")
display(pd.DataFrame({n: {p: fmt((per_seq if n == "ALL" else per_seq.loc[n])[p].values) for p in POL_ORDER}
                      for n in list(BIG) + ["ALL"]}).T)

base_only = [p for p in POL_ORDER if not p.startswith("Imitation")]
cmp_rows = []
for name in list(BIG) + ["ALL"]:
    d = per_seq if name == "ALL" else per_seq.loc[name]; sub = big if name == "ALL" else big[big.text == name]
    best = d[base_only].mean().idxmax()
    for me in POL_ORDER[:2]:
        for opp in dict.fromkeys(["H2O", best]):
            a = sub[sub.policy == me].set_index(["text", "seq", "budget"])["ratio"]
            o = sub[sub.policy == opp].set_index(["text", "seq", "budget"])["ratio"].reindex(a.index)
            try: pv = stats.wilcoxon(d[me] - d[opp]).pvalue
            except ValueError: pv = np.nan
            cmp_rows.append(dict(text=name, policy=me, vs=opp, win_rate=round((a > o).mean(), 2),
                                 mean_diff=round((a - o).mean(), 3), p_wilcoxon=pv))
print("\nPaired comparison (win_rate = share of (sequence, budget) settings won; Wilcoxon over sequences)")
display(pd.DataFrame(cmp_rows))

print("\nSeen vs UNSEEN budgets (unseen = never used in training):")
display(big.groupby(["unseen", "policy"])["ratio"].mean().unstack()[POL_ORDER].round(3))

fig, axes = plt.subplots(1, len(BIG), figsize=(5.5 * len(BIG), 4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), BIG):
    for p in POL_ORDER:
        piv = big[(big.text == name) & (big.policy == p)].pivot(index="seq", columns="budget", values="ratio")
        m = piv.mean(); imit = p.startswith("Imitation")
        ax.plot(m.index, m.values, "o", ls="-" if imit else "--", lw=2.5 if imit else 1.2, label=p)
        if imit:
            ci = np.array([boot_ci(piv[b].values) for b in piv.columns])
            ax.fill_between(m.index, ci[:, 0], ci[:, 1], alpha=.15)
    for b in UNSEEN: ax.axvline(b, color="grey", ls=":", lw=.8)
    ax.set_title(f"{name}  (dotted = unseen budget)"); ax.set_xlabel("cache budget"); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel("competitive ratio"); np.atleast_1d(axes)[-1].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.savefig("results/big_ratio.png", dpi=150); plt.show()

In [ ]:
# ---------------- Test 2: where does the gap come from? ----------------
# For every step: attention the optimum keeps minus attention the policy keeps, split by the kind of token.
# Bar height = 1 - competitive ratio. A negative piece means the policy does BETTER than the optimum on that token type
# (it pays for it elsewhere).
cols = ["lost_sink", "lost_recent", "lost_longrange"]
dec = big.groupby(["text", "policy"])[cols].mean()
fig, axes = plt.subplots(1, len(BIG), figsize=(5.5 * len(BIG), 4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), BIG):
    dec.loc[name].reindex(POL_ORDER).rename(columns=dict(lost_sink="sink tokens (first 4)", lost_recent="recent (age < k/4)",
        lost_longrange="long-range / heavy hitters")).plot.bar(stacked=True, ax=ax, legend=False, color=["tab:red", "tab:blue", "tab:orange"])
    ax.set_title(name); ax.set_xticklabels([p.replace(" (", "\n(") for p in POL_ORDER], rotation=40, ha="right", fontsize=7); ax.axhline(0, color="k", lw=.8)
np.atleast_1d(axes)[0].set_ylabel("share of the optimum's attention lost"); np.atleast_1d(axes)[-1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/big_gap_breakdown.png", dpi=150); plt.show()
display(dec.groupby("policy").mean().reindex(POL_ORDER).round(3).assign(total_gap=lambda d: d.sum(1).round(3)))

In [ ]:
# ---------------- Test 3: handed the optimum's memory ----------------
ag = big.groupby(["text", "policy"])["agree"].mean().unstack()[POL_ORDER]
chance = big["chance"].mean()
ax = ag.T.plot.bar(figsize=(9, 3.8)); ax.axhline(chance, color="k", ls="--", lw=1, label=f"random guess ({chance:.0%})")
ax.set_ylabel("drops the same token as the optimum"); ax.set_title("Given the optimum's exact cache, which token does each policy drop?")
ax.set_xticklabels([p.replace(" (", "\n(") for p in POL_ORDER], rotation=0, fontsize=7); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/big_agreement.png", dpi=150); plt.show()
display(big.groupby("policy")[["agree", "chance"]].mean().reindex(POL_ORDER).round(3))

In [ ]:
# ---------------- Test 4: stress traces (patterns the student was never trained on) ----------------
def stress_traces(T, seed=0):
    rng = np.random.default_rng(seed)
    def recency(decay=0.25):
        A = np.zeros((T, T))
        for t in range(T): A[t, :t + 1] = np.exp(-decay * (t - np.arange(t + 1)))
        return A / A.sum(1, keepdims=True)
    def blend(base, target, w):            # row-wise: (1-w)*base + w*target
        return (1 - w[:, None]) * base + w[:, None] * target
    def finish(A):
        A = np.tril(A * np.exp(rng.normal(0, .15, A.shape))); return A / A.sum(1, keepdims=True)
    base = recency(); tr = {}; steps = np.arange(T)
    tr["pure recency"] = finish(base)
    sk = np.zeros((T, T)); sk[:, :4] = .25; tr["sinks + recency"] = finish(blend(base, np.tril(sk), np.full(T, .8)))
    heavy = np.arange(20, T - 20, 40); tgt = np.zeros((T, T))
    for t in range(T):
        h = heavy[heavy < t]
        if len(h): tgt[t, h] = 1 / len(h)
    tr["periodic re-reads"] = finish(blend(base, tgt, np.where(steps % 32 < 4, .7, 0.)))
    h1, h2 = np.arange(10, 26), np.arange(T // 2 - 40, T // 2 - 24); tgt = np.zeros((T, T))
    tgt[:T // 2, h1] = 1 / len(h1); tgt[T // 2:, h2] = 1 / len(h2)
    tr["sudden topic change"] = finish(blend(base, np.tril(tgt), np.full(T, .5)))
    late = np.arange(T // 4, T // 4 + 4); tgt = np.zeros((T, T)); tgt[:, late] = .25
    tr["unpredictable late burst"] = finish(blend(base, np.tril(tgt), np.where(steps >= T - 16, .6, 0.)))
    return tr

keys = [(n, s) for n in BIG for s in range(min(4, len(BIG[n])))]
IMIT_ALL = train_imitation(big_items(keys), name="Imitation (trained on real text)")
ST_POLS = lambda: [IMIT_ALL, HeavyHitter(0.5, "H2O"), Hybrid(), StreamingLLM(), SlidingWindow(), HeavyHitter(0.0, "HeavyHitter-only")]
st_rows = []
for seed in range(3):
    for tname, A in stress_traces(T, seed).items():
        for b in [0.1, 0.25]:
            k = int(b * T); end = optimal_schedule(A, k)[0]
            opt_m = simulate(A, k, ScheduledPolicy(end))[0][k:].mean()
            for p in ST_POLS():
                st_rows.append(dict(trace=tname, budget=b, seed=seed, policy=p.name, ratio=simulate(A, k, p)[0][k:].mean() / opt_m))
st = pd.DataFrame(st_rows); st.to_csv("results/stress_traces.csv", index=False)
tab4 = st.groupby(["trace", "policy"])["ratio"].mean().unstack()
tab4 = tab4[[c for c in ["Imitation (trained on real text)", "H2O", "Hybrid", "StreamingLLM", "SlidingWindow", "HeavyHitter-only"]]]
tab4.loc["WORST CASE"] = tab4.min()
fig, ax = plt.subplots(figsize=(9, 3.8)); im = ax.imshow(tab4.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(tab4.shape[1])); ax.set_xticklabels([c.replace(" (", "\n(") for c in tab4.columns], fontsize=8)
ax.set_yticks(range(tab4.shape[0])); ax.set_yticklabels(tab4.index)
for i in range(tab4.shape[0]):
    for j in range(tab4.shape[1]): ax.text(j, i, f"{tab4.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Competitive ratio on synthetic stress traces (green = near optimum)"); plt.colorbar(im); plt.tight_layout()
plt.savefig("results/stress_traces.png", dpi=150); plt.show()

In [ ]:
# ---------------- Test 5a: real perplexity at scale (one UNSEEN budget, several sequences) ----------------
PPL_B = 0.15; kk = int(PPL_B * T); rat = big.set_index(["text", "seq", "budget", "policy"])["ratio"]
rows7 = []; t0 = time.time()
for name in BIG:
    for si in range(min(CFG["n_ppl"], len(BIG[name]))):
        ids = BIG[name][si]; A, nll_full = A_BIG[(name, si)]; ppl_full = np.exp(nll_full[kk:].mean())
        for p in [BIG_X[name], HeavyHitter(0.5, "H2O"), Hybrid(), StreamingLLM(), SlidingWindow(), ScheduledPolicy(OPT_BIG[(name, si, PPL_B)])]:
            nll, _ = run_with_eviction(model, ids, kk, p); ppl = np.exp(nll[kk:].mean())
            rows7.append(dict(text=name, seq=si, policy=p.name, dppl_pct=100 * (ppl / ppl_full - 1),
                              ratio=rat.get((name, si, PPL_B, p.name), 1.0)))
        print(f"{name} seq {si} done ({time.time() - t0:.0f}s)")
df7 = pd.DataFrame(rows7); df7.to_csv("results/big_ppl.csv", index=False)
P7 = ["Imitation (cross-domain)", "H2O", "Hybrid", "StreamingLLM", "SlidingWindow", "Optimal (offline)"]
piv7 = df7.pivot_table(index=["text", "seq"], columns="policy", values="dppl_pct")[P7]
print(f"dPPL % vs full cache at k = {PPL_B:.0%} of tokens (lower is better), mean [95% bootstrap CI]:")
display(pd.Series({p: f"{piv7[p].mean():.1f} [{boot_ci(piv7[p].values)[0]:.1f}, {boot_ci(piv7[p].values)[1]:.1f}]" for p in P7}).to_frame("dPPL %"))
try: pv = stats.wilcoxon(piv7[P7[0]] - piv7["H2O"]).pvalue
except ValueError: pv = np.nan
print(f"Imitation beats H2O on perplexity in {(piv7[P7[0]] < piv7['H2O']).mean():.0%} of sequences (Wilcoxon p = {pv:.3g}, n = {len(piv7)})")
rho, pr = stats.spearmanr(df7.ratio, df7.dppl_pct)
print(f"Is attention retained a good proxy?  Spearman rho(ratio, dPPL) = {rho:.2f} (p = {pr:.1e}); strongly negative = yes")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
m = piv7.mean(); err = np.array([np.abs(boot_ci(piv7[p].values) - piv7[p].mean()) for p in P7]).T
axes[0].bar(range(len(P7)), m.values, yerr=err, capsize=4); axes[0].set_xticks(range(len(P7))); axes[0].set_xticklabels([p.replace(" (", "\n(") for p in P7], fontsize=7)
axes[0].set_ylabel("perplexity increase vs full cache (%)"); axes[0].set_title("Real quality loss (lower = better)")
for p in P7:
    d = df7[df7.policy == p]; axes[1].scatter(d.ratio, d.dppl_pct, label=p)
axes[1].set_xlabel("attention retained (competitive ratio)"); axes[1].set_ylabel("dPPL %"); axes[1].set_yscale("symlog"); axes[1].legend(fontsize=7)
axes[1].set_title(f"Proxy check: Spearman rho = {rho:.2f}"); plt.tight_layout(); plt.savefig("results/big_ppl.png", dpi=150); plt.show()

In [ ]:
# ---------------- Test 5b: needle in a haystack, repeated (needs section 9 to have run: uses tok, filler, q_ids) ----------------
def wilson(s, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = s / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den; return max(0.0, c - h), min(1.0, c + h)

N_DEPTHS = [0.1, 0.3, 0.5, 0.7, 0.9]; NB = 0.2; kn = int(NB * T)
rng = np.random.default_rng(SEED + 1); rows8 = []; t0 = time.time()
for rep in range(CFG["n_reps"]):
    for depth in N_DEPTHS:
        code_str = str(rng.integers(1000, 9999))                 # new code and new filler text every repetition
        needle = tok.encode(f" The secret code is {code_str}. ", add_special_tokens=False)
        n_fill = T - len(needle) - len(q_ids); pos = int(depth * n_fill); f = filler[rep * 700: rep * 700 + n_fill + 5]
        ids = torch.tensor(f[:pos] + needle + f[pos:n_fill] + q_ids)
        for p in [BIG_X["wikipedia"], HeavyHitter(0.5, "H2O"), Hybrid(), StreamingLLM(), SlidingWindow()]:
            _, gen = run_with_eviction(model, ids, kn, p, gen_tokens=6)
            rows8.append(dict(rep=rep, depth=depth, policy=p.name, success=code_str in tok.decode(gen).replace(" ", "")))
    print(f"repetition {rep + 1}/{CFG['n_reps']} done ({time.time() - t0:.0f}s)")
df8 = pd.DataFrame(rows8); df8.to_csv("results/big_needle.csv", index=False)
heat = df8.pivot_table(index="policy", columns="depth", values="success", aggfunc="mean")
cnt = df8.pivot_table(index="policy", columns="depth", values="success", aggfunc="sum").astype(int)
fig, ax = plt.subplots(figsize=(7, 3.2)); im = ax.imshow(heat.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels([f"{d:.0%}" for d in heat.columns]); ax.set_yticks(range(len(heat.index))); ax.set_yticklabels(heat.index, fontsize=8)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]): ax.text(j, i, f"{cnt.values[i, j]}/{CFG['n_reps']}", ha="center", va="center", fontsize=9)
ax.set_xlabel("needle depth (0% = far back, 100% = near the question)"); ax.set_title(f"Needle retrieval at k = {NB:.0%} of tokens"); plt.tight_layout()
plt.savefig("results/big_needle.png", dpi=150); plt.show()
acc8 = df8.groupby("policy")["success"].agg(["sum", "count"])
acc8["acc"] = acc8["sum"] / acc8["count"]; acc8[["ci_lo", "ci_hi"]] = [wilson(s, n) for s, n in zip(acc8["sum"], acc8["count"])]
display(acc8.round(2))

In [ ]:
print("=== 10c: numbers to quote ===")
ps = per_seq.mean(); I, H = ps[POL_ORDER[0]], ps["H2O"]
print(f"[1] Mean ratio over {len(per_seq)} sequences x {len(TEST_BUDGETS)} budgets: cross-domain imitation {I:.3f}, in-domain {ps[POL_ORDER[1]]:.3f}, H2O {H:.3f}, best hand-designed: {ps[base_only].idxmax()} {ps[base_only].max():.3f}")
un = big[big.unseen].groupby("policy")["ratio"].mean()
print(f"    On UNSEEN budgets only: imitation {un[POL_ORDER[0]]:.3f} vs H2O {un['H2O']:.3f}")
g = dec.groupby("policy").mean().reindex(POL_ORDER)
print("[2] Biggest source of lost attention per policy:", {p: g.loc[p].idxmax().replace('lost_', '') for p in POL_ORDER})
a = big.groupby("policy")["agree"].mean()
print(f"[3] Same-token-as-optimum rate: imitation {a[POL_ORDER[0]]:.0%}, H2O {a['H2O']:.0%}, chance {big['chance'].mean():.0%}")
w = tab4.loc["WORST CASE"]; print(f"[4] Worst-case stress ratio: " + ", ".join(f"{c} {v:.2f}" for c, v in w.items()))
print(f"[5] dPPL%: " + ", ".join(f"{p} {piv7[p].mean():.1f}" for p in P7) + f" | Spearman(ratio, dPPL) = {rho:.2f}")
print("    Needle accuracy: " + ", ".join(f"{p} {r.acc:.0%}" for p, r in acc8.iterrows()))
print("\nIf imitation does not win a test, say so: a clean negative result with this much evidence is still a finding.")

## 10d. DAgger: train the student on its OWN mistakes
**Problem with plain imitation.** The student is trained on states the teacher (and a few hand-made policies) create. At test time it faces states created by *its own* earlier eviction mistakes: a distribution mismatch that makes errors compound.

**DAgger fix** (Ross et al., 2011): run the current student, record the states *it* visits, let the **exact min-cost-flow optimum relabel them** (labels come straight from the schedule `end[j]` we already have, no new solves), add them to the training set, retrain, repeat.

| # | Test | Question |
|---|------|----------|
| 1 | **Rounds curve** | Does the competitive ratio improve with each DAgger round, on a text type never trained on? |
| 2 | **Error build-up over time** | Do errors compound within a sequence, and does DAgger flatten that? |
| 3 | **Own-state accuracy** | Is the student worse on states it visits itself than on the teacher's states, and does DAgger close that gap? |
| 4 | **Data efficiency** | Does DAgger need fewer training sequences? |
| 5 | **What to imitate** | Does the label horizon (keep for k/4, k/2 or k more steps) matter? |

All students are cross-domain: tested only on a text type they never trained on. Needs sections 10c.0 and 10c.1 (data and optimum) to have run.

In [ ]:
from src.imitation import *            # includes dagger_train and state_auc
ROUNDS = 2 if PRESET == "quick" else 4
EVAL_B = [0.1, 0.15, 0.3]            # budgets for the cheaper tests (all inside TEST_BUDGETS)

# optimum's per-sequence score, computed once (denominator of every ratio)
OPTM = {}
for (n, s), (A, _) in A_BIG.items():
    for b in TEST_BUDGETS:
        k = int(b * T); OPTM[(n, s, b)] = simulate(A, k, ScheduledPolicy(OPT_BIG[(n, s, b)]))[0][k:].mean()

def eval_ratios(pol, name, budgets):
    """Competitive ratio per sequence (averaged over budgets) on held-out text type `name`."""
    out = []
    for s in range(len(BIG[name])):
        A = A_BIG[(name, s)][0]
        out.append(np.mean([simulate(A, int(b * T), pol)[0][int(b * T):].mean() / OPTM[(name, s, b)] for b in budgets]))
    return np.array(out)

def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); r = np.random.default_rng(seed)
    return np.percentile(r.choice(x, (n, len(x))).mean(1), [2.5, 97.5])

DG = {}                                # DG[text type] = [student round 0, 1, ..., ROUNDS], trained on the OTHER types
t0 = time.time()
for name in BIG:
    keys = [(n, s) for n in BIG if n != name for s in range(len(BIG[n]))]
    DG[name] = dagger_train(big_items(keys), rounds=ROUNDS, name="DAgger")
    print(f"test={name:10s}: {ROUNDS} DAgger rounds, final training set {DG[name][-1].n_train:,} rows  ({time.time() - t0:.0f}s)")

In [ ]:
# ---------------- Test 1: rounds curve ----------------
from scipy import stats
BASES = {"H2O": HeavyHitter(0.5, "H2O"), "Hybrid": Hybrid(), "StreamingLLM": StreamingLLM(), "SlidingWindow": SlidingWindow()}
res = {nm: np.array([eval_ratios(DG[nm][r], nm, TEST_BUDGETS) for r in range(ROUNDS + 1)]) for nm in BIG}   # [round, sequence]
base = {nm: {b: eval_ratios(p, nm, TEST_BUDGETS) for b, p in BASES.items()} for nm in BIG}

fig, axes = plt.subplots(1, len(BIG), figsize=(5 * len(BIG), 4), sharey=True)
for ax, nm in zip(np.atleast_1d(axes), BIG):
    m = res[nm].mean(1); ci = np.array([boot_ci(res[nm][r]) for r in range(ROUNDS + 1)])
    ax.plot(range(ROUNDS + 1), m, "o-", lw=2.5, label="DAgger student"); ax.fill_between(range(ROUNDS + 1), ci[:, 0], ci[:, 1], alpha=.2)
    for b, c in zip(["H2O", "SlidingWindow"], ["tab:orange", "tab:green"]):
        ax.axhline(base[nm][b].mean(), color=c, ls="--", label=b)
    ax.set_title(f"test on {nm}"); ax.set_xlabel("DAgger round (0 = plain imitation)"); ax.set_xticks(range(ROUNDS + 1)); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel("competitive ratio"); np.atleast_1d(axes)[-1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/dagger_rounds.png", dpi=150); plt.show()

allr = np.concatenate([res[nm] for nm in BIG], axis=1)            # [round, all sequences]
tab = pd.DataFrame({"mean ratio": allr.mean(1), "CI low": [boot_ci(allr[r])[0] for r in range(ROUNDS + 1)],
                    "CI high": [boot_ci(allr[r])[1] for r in range(ROUNDS + 1)]}, index=[f"round {r}" for r in range(ROUNDS + 1)])
display(tab.round(3))
try: pv = stats.wilcoxon(allr[-1] - allr[0]).pvalue
except ValueError: pv = np.nan
print(f"Final round vs plain imitation: {(allr[-1] > allr[0]).mean():.0%} of sequences improve, mean change {np.mean(allr[-1] - allr[0]):+.4f}, Wilcoxon p = {pv:.3g}  (n = {allr.shape[1]})")
print("Hand-designed baselines (mean over the same sequences):", {b: round(float(np.mean(np.concatenate([base[nm][b] for nm in BIG]))), 3) for b in BASES})

In [ ]:
# ---------------- Test 2: do errors compound over the sequence? ----------------
B2 = 0.15; k2 = int(B2 * T)                                         # an unseen budget
LAB0, LAB1 = "plain imitation (round 0)", f"DAgger (round {ROUNDS})"
cum = {LAB0: [], LAB1: [], "H2O": [], "SlidingWindow": []}
for nm in BIG:
    for s in range(len(BIG[nm])):
        A = A_BIG[(nm, s)][0]; opt = simulate(A, k2, ScheduledPolicy(OPT_BIG[(nm, s, B2)]))[0]
        for lab, pol in [(LAB0, DG[nm][0]), (LAB1, DG[nm][-1]), ("H2O", HeavyHitter(0.5, "H2O")), ("SlidingWindow", SlidingWindow())]:
            cum[lab].append(np.cumsum((opt - simulate(A, k2, pol)[0])[k2:]))      # running total of attention lost vs optimum
x = np.arange(k2, T); fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for lab, c in cum.items():
    c = np.array(c); m = c.mean(0); se = c.std(0) / np.sqrt(len(c))
    axes[0].plot(x, m, label=lab, lw=2.5 if "imitation" in lab or "DAgger" in lab else 1.2, ls="-" if ("imitation" in lab or "DAgger" in lab) else "--")
    axes[0].fill_between(x, m - se, m + se, alpha=.15)
axes[0].set_xlabel("step t"); axes[0].set_ylabel("cumulative attention lost vs optimum"); axes[0].set_title("Error build-up (flatter = better)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
comp = {}
for lab, c in cum.items():
    g = np.diff(np.concatenate([np.zeros((len(c), 1)), np.array(c)], axis=1), axis=1); h = g.shape[1] // 2
    comp[lab] = g[:, h:].mean() / max(g[:, :h].mean(), 1e-12)
axes[1].bar(range(len(comp)), list(comp.values())); axes[1].axhline(1, color="k", ls="--", lw=1)
axes[1].set_xticks(range(len(comp))); axes[1].set_xticklabels([l.replace(" (", "\n(") for l in comp], fontsize=7)
axes[1].set_ylabel("loss per step, 2nd half / 1st half"); axes[1].set_title("> 1 means errors compound as the sequence goes on")
plt.tight_layout(); plt.savefig("results/dagger_error_buildup.png", dpi=150); plt.show()
print({k: round(float(v), 2) for k, v in comp.items()})

In [ ]:
# ---------------- Test 3: accuracy on the student's OWN states vs the teacher's states ----------------
rows = []
for nm in BIG:
    its = [(A_BIG[(nm, s)][0], int(b * T), OPT_BIG[(nm, s, b)]) for s in range(len(BIG[nm])) for b in EVAL_B]
    for r, stu in enumerate(DG[nm]):
        rows.append(dict(text=nm, round=r, teacher_states=state_auc(stu, its, own=False), own_states=state_auc(stu, its, own=True)))
d3 = pd.DataFrame(rows); d3["mismatch"] = d3.teacher_states - d3.own_states
m3 = d3.groupby("round")[["teacher_states", "own_states", "mismatch"]].mean()
display(m3.round(3))
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(m3.index, m3.teacher_states, "o-", label="on the teacher's states"); ax.plot(m3.index, m3.own_states, "s-", label="on its OWN states (what it faces at test time)")
ax.set_xlabel("DAgger round"); ax.set_ylabel("AUC at ranking keep vs drop (0.5 = chance, 1 = perfect)"); ax.set_xticks(m3.index); ax.grid(alpha=.3); ax.legend(fontsize=8)
ax.set_title("Distribution mismatch: does the gap close?"); plt.tight_layout(); plt.savefig("results/dagger_mismatch.png", dpi=150); plt.show()

In [ ]:
# ---------------- Test 4: data efficiency ----------------
sizes = [1, 2, 4, 8] if PRESET == "full" else [1, 2]; n_seeds = 2 if PRESET == "full" else 1; rnds = 3 if PRESET == "full" else 1
rows = []; t0 = time.time()
for nm in BIG:
    for n_tr in sizes:
        for seed in range(n_seeds):
            r = np.random.default_rng(seed); keys = []
            for n in BIG:
                if n != nm: keys += [(n, int(i)) for i in r.choice(len(BIG[n]), min(n_tr, len(BIG[n])), replace=False)]
            stus = dagger_train(big_items(keys), rounds=rnds, seed=seed)
            for lab, stu in [("plain imitation", stus[0]), ("DAgger", stus[-1])]:
                rows.append(dict(text=nm, n_train=len(keys), seed=seed, variant=lab, ratio=eval_ratios(stu, nm, EVAL_B).mean()))
    print(f"{nm} done ({time.time() - t0:.0f}s)")
d4 = pd.DataFrame(rows); d4.to_csv("results/dagger_data_efficiency.csv", index=False)
h2o_ref = np.mean([eval_ratios(HeavyHitter(0.5, "H2O"), nm, EVAL_B).mean() for nm in BIG])
fig, ax = plt.subplots(figsize=(6.5, 4))
for lab in ["plain imitation", "DAgger"]:
    g = d4[d4.variant == lab].groupby("n_train")["ratio"]; m = g.mean(); ci = np.array([boot_ci(g.get_group(n).values) for n in m.index])
    ax.plot(m.index, m.values, "o-", label=lab); ax.fill_between(m.index, ci[:, 0], ci[:, 1], alpha=.2)
ax.axhline(h2o_ref, color="tab:orange", ls="--", label="H2O"); ax.set_xscale("log", base=2)
ax.set_xlabel("training sequences (from the other text types)"); ax.set_ylabel("competitive ratio on held-out text type"); ax.grid(alpha=.3); ax.legend(fontsize=8)
ax.set_title("Data efficiency"); plt.tight_layout(); plt.savefig("results/dagger_data_efficiency.png", dpi=150); plt.show()
display(d4.pivot_table(index="n_train", columns="variant", values="ratio").round(3))

In [ ]:
# ---------------- Test 5: what should the student imitate? (label horizon) ----------------
# label = 1 if the optimum still holds the token h * k steps from now: short horizon = "is it needed soon", long = "is it needed for a long time"
rows = []
for hf in [0.25, 0.5, 1.0]:
    for nm in BIG:
        keys = [(n, s) for n in BIG if n != nm for s in range(len(BIG[n]))]
        stu = dagger_train(big_items(keys), rounds=rnds, h_frac=hf, seed=0)[-1]
        rows.append(dict(horizon=hf, text=nm, ratio=eval_ratios(stu, nm, EVAL_B).mean()))
    print(f"horizon {hf} done")
d5 = pd.DataFrame(rows); d5.to_csv("results/dagger_horizon.csv", index=False)
display(d5.pivot(index="horizon", columns="text", values="ratio").assign(mean=lambda d: d.mean(1)).round(3))
ax = d5.groupby("horizon")["ratio"].mean().plot.bar(figsize=(5, 3.5), rot=0); ax.set_ylabel("mean competitive ratio"); ax.set_xlabel("label horizon (x k steps)")
ax.set_ylim(d5.ratio.min() - .02, d5.ratio.max() + .02); plt.tight_layout(); plt.savefig("results/dagger_horizon.png", dpi=150); plt.show()

In [ ]:
print("=== 10d: DAgger numbers to quote ===")
print(f"[1] Ratio by round (all held-out sequences): " + ", ".join(f"r{r} {allr[r].mean():.3f}" for r in range(ROUNDS + 1)) + f"  | Wilcoxon final vs r0: p = {pv:.3g}")
print(f"    Best hand-designed baseline: " + max(((b, np.mean(np.concatenate([base[nm][b] for nm in BIG]))) for b in BASES), key=lambda x: x[1]).__repr__())
print(f"[2] Loss-per-step growth (2nd half / 1st half): " + ", ".join(f"{k} {v:.2f}" for k, v in comp.items()))
print(f"[3] Mismatch (AUC on teacher states minus own states): round 0 {m3.mismatch.iloc[0]:.3f} -> round {ROUNDS} {m3.mismatch.iloc[-1]:.3f}")
p4 = d4.pivot_table(index="n_train", columns="variant", values="ratio"); print("[4] Data efficiency (ratio): "); print(p4.round(3).to_string())
print("[5] Best label horizon:", d5.groupby("horizon")["ratio"].mean().idxmax(), "x k")
print("\nIf DAgger does not help, that is a result too: it would mean the simple students already see representative states.")

## 10e. A new hybrid — **Chameleon** — and the **Gauntlet** (one battery for every policy)

### Chameleon (what it is, in viva words)
No single eviction signal wins on every text: recency wins on dialogue, accumulated attention on long documents, re-read frequency on code. H2O hard-codes a 50/50 mix, Hybrid a 25/75 mix. **Chameleon learns the mix while it runs.** It keeps five *experts* (each is a classic rule: **recency** ≈ LRU/FIFO, **cumulative attention** ≈ H2O, **last-step attention** ≈ TOVA, **slow-EMA attention** ≈ Scissorhands-style persistence, **hit-rate** ≈ LFU), scores each expert every step on the attention that has *just* arrived, and re-weights them with **Fixed-Share Hedge** (multiplicative weights + a small share so it can switch when the text changes). Decision = evict the token with the lowest weighted-average rank. Sinks and the newest few tokens are protected.

* **Online and training-free:** no offline teacher, no leakage, no distribution shift between training and test. Contrast with Imitation/DAgger, which learn offline from the optimum.
* **Guarantee (honest version):** the *weights* have regret ≤ ln N/η + ηT/8 + T·ln(1/(1−α))/η against the best fixed expert, on any loss sequence, even when losses depend on our own past evictions (proof in the docstring; checked on every run by `regret_check`). The loss is a surrogate (ranking quality on observed attention), **not** the competitive ratio, and rank-averaging is a heuristic realisation of the weights. Do not claim a competitive-ratio guarantee.
* **Cost:** O(N·k log k) per step, N = 5.

### The Gauntlet
Every policy is wrapped in a `Spec`; learned policies are trained **only** on the other text types. Adding a policy = adding one `Spec`; no test mentions a specific policy.

| # | Test | Question it answers |
|---|------|---------------------|
| 0 | **Self-tests** | Optimum = brute force? No policy beats it? Capacity respected? **No policy peeks at the future** (future rows are scrambled; past decisions must not change)? Regret bound holds? |
| 1 | **Scoreboard** | Competitive ratio on held-out text and **unseen budgets**: mean, bootstrap CI, worst text, 5th percentile, mean rank, pairwise win-rates, Holm-corrected Wilcoxon. Only **TEST** sequences (odd index); DEV (even) is for tuning only. |
| 2 | **Distribution shift** | Real model attention on **spliced** sequences (wiki → code → dialogue …) never seen in training: does the policy adapt mid-sequence? |
| 3 | **Adversary** | Hill-climb to *find* an input that hurts each policy most: an empirical upper bound on its competitive ratio. |
| 4 | **Stress traces** | The five hand-built attention patterns of section 10c, for every policy. |
| 5 | **Real quality** | Perplexity with a physically pruned cache + needle retrieval, then: **does attention-retained rank the policies the same way perplexity does** (Kendall τ)? |
| 6 | **Ablations** | Which part of Chameleon matters (equal weights, no fixed-share, no sinks, leave-one-expert-out, ...) on the same battery. |
| 7 | **Overhead** | µs per decision and peak memory. |

**Pre-registered decision rule** (in `verdict`): primary metric = ratio on unseen budgets, pure text, TEST sequences. A policy wins only if the paired CI of the gap to the runner-up excludes 0; otherwise report a **tie**. Robustness (worst text, adversarial worst case) is printed next to it. Whatever comes out, report it: a tie or a loss is a result.

Needs sections 10c.0 and 10c.1 (data `A_BIG`, optima `OPT_BIG`) and, for test 5, section 9 (`filler`, `q_ids`). Set `RUN_LONG = True` below on the server for a 2T-length generalisation test.

In [ ]:
# Chameleon: online-learning hybrid eviction policy
# Source: src/chameleon.py
from src.chameleon import *


In [ ]:
# Gauntlet: unified test battery for every eviction policy
# Source: src/gauntlet.py
from src.gauntlet import *


In [ ]:
# ---- Test 0: self-tests (seconds). Do not trust any later number if this fails. ----
import importlib
import src.chameleon, src.gauntlet; importlib.reload(src.chameleon); importlib.reload(src.gauntlet)
from src.chameleon import *
from src.gauntlet import *
selftest()

In [ ]:
# ---- assemble the data every test uses ----
RUN_LONG = False          # True on the server: adds a 2T-token generalisation test (optimum is O(T^2) arcs; needs ortools)
N_SPLICE = 4 if PRESET == "full" else 2      # spliced sequences per ordering

DATA = {key: A for key, (A, _) in A_BIG.items()}        # real attention, pure text types (from 10c)
NLL  = {key: nll for key, (_, nll) in A_BIG.items()}
KIND = {n: "pure" for n in BIG}
OPT  = dict(OPT_BIG); OPTM = {}
def add_optima(keys, budgets):
    for (n, s) in keys:
        A = DATA[(n, s)]; Tn = A.shape[0]
        for b in budgets:
            k = int(b * Tn)
            if (n, s, b) not in OPT: OPT[(n, s, b)] = get_optimum(A, k)
            OPTM[(n, s, b)] = simulate(A, k, ScheduledPolicy(OPT[(n, s, b)]))[0][k:].mean()
add_optima(list(DATA), TEST_BUDGETS)
print(len(DATA), "pure sequences,", len(OPT), "optimal schedules")

In [ ]:
# ---- Test 2 data: SPLICED sequences, real attention (needs the model). Chunks never used in BIG, so no overlap with training. ----
SEG = T // 3; SPLICE_IDS = {}
def free_chunks(name):
    ids = tok(TEXTS[name], return_tensors="pt", add_special_tokens=False).input_ids[0]
    n_avail = len(ids) // T
    used = set(np.unique(np.linspace(0, n_avail - 1, min(CFG["n_seq"], n_avail)).round().astype(int)).tolist())
    free = [p for p in range(n_avail) if p not in used] or list(range(n_avail))
    return ids, free
FREE = {n: free_chunks(n) for n in BIG}
names = list(BIG)
orders = [tuple(names[(i + j) % len(names)] for j in range(min(3, len(names)))) for i in range(len(names))] if len(names) > 1 else []
for order in orders:
    tname = "shift: " + ">".join(o[:4] for o in order)
    for i in range(N_SPLICE):
        parts = []
        for pos, nm in enumerate(order):
            ids, free = FREE[nm]; p = free[(i * len(order) + pos) % len(free)]
            L = SEG if pos < len(order) - 1 else T - SEG * (len(order) - 1)
            parts.append(ids[p * T: p * T + L])
        sid = torch.cat(parts); assert len(sid) == T
        A, nll = attention_matrix(model, sid); DATA[(tname, i)] = A; NLL[(tname, i)] = nll; SPLICE_IDS[(tname, i)] = sid
    KIND[tname] = "shift"
add_optima([k for k in DATA if KIND[k[0]] == "shift"], TEST_BUDGETS)
print("shift texts:", [n for n in KIND if KIND[n] == "shift"])

In [ ]:
# ---- Test 1 + 2: the scoreboard (every policy, identical protocol) ----
SPECS = default_specs(dagger_rounds=3 if PRESET == "full" else 1)
t0 = time.time()
df_all = run_ratio(DATA, OPT, OPTM, KIND, SPECS, TEST_BUDGETS, TRAIN_BUDGETS)
df_all.to_csv("results/gauntlet_ratio.csv", index=False)
SCORE_CACHE = run_ratio.cache          # the trained policies, reused by the perplexity and needle tests below
test = df_all[df_all.split == "test"]
print(f"{len(df_all)} runs in {time.time() - t0:.0f}s   (reporting on TEST sequences only: {test.groupby(['text']).seq.nunique().to_dict()})")
cols = ["mean ratio", "CI lo", "CI hi", "worst text", "worst-text name", "p5 over settings", "mean rank", "us/step"]
print("\n=== pure text, all budgets ===");               display(leaderboard(test[test.kind == "pure"])[cols].round(3))
print("\n=== pure text, UNSEEN budgets only (primary) ==="); display(leaderboard(test[(test.kind == "pure") & test.unseen])[cols].round(3))
if (test.kind == "shift").any():
    print("\n=== DISTRIBUTION SHIFT: spliced sequences ==="); display(leaderboard(test[test.kind == "shift"])[cols].round(3))
print("\n=== per text type (mean ratio, TEST) ===");        display(test.pivot_table(index="policy", columns="text", values="ratio").round(3))
print("\n=== per budget (mean ratio, pure TEST) ===");      display(test[test.kind == "pure"].pivot_table(index="policy", columns="budget", values="ratio").round(3))
lead = leaderboard(test[(test.kind == "pure") & test.unseen]).index[0]
print(f"\n=== every policy vs the leader ({lead}), paired per sequence ==="); display(vs_leader(test[test.kind == "pure"], lead).round(4))

lb = leaderboard(test[test.kind == "pure"]); ci = lb[["CI lo", "CI hi"]].values.T
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].barh(lb.index[::-1], lb["mean ratio"][::-1], xerr=np.abs(ci[:, ::-1] - lb["mean ratio"].values[::-1]), capsize=3)
axes[0].set_xlim(max(0, lb["mean ratio"].min() - .1), 1.0); axes[0].set_xlabel("competitive ratio (95% bootstrap CI over sequences)"); axes[0].set_title("Scoreboard: pure text, TEST sequences")
W = pairwise(test); W = W.loc[lb.index, lb.index]; im = axes[1].imshow(W.values, cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_xticks(range(len(W))); axes[1].set_xticklabels(W.columns, rotation=45, ha="right", fontsize=8); axes[1].set_yticks(range(len(W))); axes[1].set_yticklabels(W.index, fontsize=8)
for i in range(len(W)):
    for j in range(len(W)): axes[1].text(j, i, f"{W.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
axes[1].set_title("Win-rate: row beats column (share of settings)"); plt.tight_layout(); plt.savefig("results/gauntlet_scoreboard.png", dpi=150); plt.show()

In [ ]:
# ---- Test 3 (adversary) and Test 4 (stress traces) ----
POOL_ALL = train_pool(DATA, OPT, KIND, "__none__", TRAIN_BUDGETS)      # learned policies: trained on all real pure text
print("Adversary: hill-climb for the input where each policy is worst (T=64, k=16). Lower ratio = easier to break.")
df_adv = run_adversary(SPECS, POOL_ALL, T=64, k=16, iters=150 if PRESET == "full" else 60, restarts=3 if PRESET == "full" else 1)
df_adv.to_csv("results/gauntlet_adversary.csv", index=False)
display(df_adv.sort_values("worst_ratio", ascending=False).round(3).set_index("policy"))

df_st = run_stress(SPECS, stress_traces, T, POOL_ALL)                 # stress_traces is defined in section 10c (Test 4)
df_st.to_csv("results/gauntlet_stress.csv", index=False)
tab_st = df_st.groupby(["trace", "policy"])["ratio"].mean().unstack(); tab_st.loc["WORST CASE"] = tab_st.min()
fig, ax = plt.subplots(figsize=(11, 3.8)); im = ax.imshow(tab_st.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(tab_st.shape[1])); ax.set_xticklabels(tab_st.columns, rotation=30, ha="right", fontsize=8); ax.set_yticks(range(tab_st.shape[0])); ax.set_yticklabels(tab_st.index)
for i in range(tab_st.shape[0]):
    for j in range(tab_st.shape[1]): ax.text(j, i, f"{tab_st.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Stress traces: competitive ratio (green = near optimum)"); plt.colorbar(im); plt.tight_layout(); plt.savefig("results/gauntlet_stress.png", dpi=150); plt.show()

In [ ]:
# ---- Test 5a: real perplexity, physically pruned cache, EVERY policy ----
PPL_B = 0.15; kk = int(PPL_B * T); rows = []; t0 = time.time()
for name in [n for n in KIND if KIND[n] != "long"]:
    keys = [k for k in DATA if k[0] == name and split_of(k[1]) == "test"][:CFG["n_ppl"] if KIND[name] == "pure" else 2]
    made = SCORE_CACHE[pool_names_for(KIND, name)]                    # the SAME trained models used in the scoreboard
    for (n, si) in keys:
        ids = BIG[n][si] if KIND[n] == "pure" else SPLICE_IDS[(n, si)]
        ppl_full = np.exp(NLL[(n, si)][kk:].mean())
        for sp in SPECS:
            nll, _ = run_with_eviction(model, ids, kk, made[sp.name]()); ppl = np.exp(nll[kk:].mean())
            rows.append(dict(text=n, kind=KIND[n], seq=si, policy=sp.name, dppl_pct=100 * (ppl / ppl_full - 1)))
    print(f"{name} done ({time.time() - t0:.0f}s)")
df_ppl = pd.DataFrame(rows); df_ppl.to_csv("results/gauntlet_ppl.csv", index=False)
pv = df_ppl.pivot_table(index=["text", "seq"], columns="policy", values="dppl_pct")
tab = pd.DataFrame({"dPPL % mean": pv.mean(), "CI lo": [boot_ci(pv[p].values)[0] for p in pv], "CI hi": [boot_ci(pv[p].values)[1] for p in pv],
                    "worst sequence": pv.max(), "win-rate vs H2O": [(pv[p] < pv["H2O"]).mean() for p in pv]}).sort_values("dPPL % mean")
print(f"dPPL % vs full cache at k = {PPL_B:.0%} (lower is better; n = {len(pv)} sequences)"); display(tab.round(2))
# does attention-retained rank the policies the way perplexity does?
rat = test[np.isclose(test.budget, PPL_B)].groupby("policy")["ratio"].mean()
tau, p_tau, common = proxy_validity(rat, pv.mean())
print(f"Kendall tau between 'ranking by attention retained' and 'ranking by perplexity' over {len(common)} policies: {tau:+.2f} (p = {p_tau:.3f})  [+1 = the proxy ranks policies correctly]")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for p in common: ax.scatter(rat[p], pv.mean()[p], s=60); ax.annotate(p, (rat[p], pv.mean()[p]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("attention retained (competitive ratio)"); ax.set_ylabel("perplexity increase % (lower = better)"); ax.set_yscale("symlog"); ax.grid(alpha=.3)
ax.set_title(f"Proxy validity across policies: Kendall tau = {tau:+.2f}"); plt.tight_layout(); plt.savefig("results/gauntlet_proxy.png", dpi=150); plt.show()

In [ ]:
# ---- Test 5b: needle in a haystack, EVERY policy, repeated (filler is Wikipedia -> learned policies use the pool without Wikipedia) ----
def wilson(s, n, z=1.96):
    p = s / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)
made_w = SCORE_CACHE[pool_names_for(KIND, "wikipedia")]
kn = int(0.2 * T); rng = np.random.default_rng(SEED + 7); rows = []; t0 = time.time()
for rep in range(CFG["n_reps"]):
    for depth in [0.1, 0.3, 0.5, 0.7, 0.9]:
        code_str = str(rng.integers(1000, 9999)); needle = tok.encode(f" The secret code is {code_str}. ", add_special_tokens=False)
        n_fill = T - len(needle) - len(q_ids); pos = int(depth * n_fill); f = filler[rep * 700: rep * 700 + n_fill + 5]
        ids = torch.tensor(f[:pos] + needle + f[pos:n_fill] + q_ids)
        for sp in SPECS:
            _, gen = run_with_eviction(model, ids, kn, made_w[sp.name](), gen_tokens=6)
            rows.append(dict(rep=rep, depth=depth, policy=sp.name, success=code_str in tok.decode(gen).replace(" ", "")))
    print(f"repetition {rep + 1}/{CFG['n_reps']} done ({time.time() - t0:.0f}s)")
df_ndl = pd.DataFrame(rows); df_ndl.to_csv("results/gauntlet_needle.csv", index=False)
acc = df_ndl.groupby("policy")["success"].agg(["sum", "count"]); acc["acc"] = acc["sum"] / acc["count"]
acc[["CI lo", "CI hi"]] = [wilson(s, n) for s, n in zip(acc["sum"], acc["count"])]
display(acc.sort_values("acc", ascending=False).round(2))
heat = df_ndl.pivot_table(index="policy", columns="depth", values="success", aggfunc="mean")
fig, ax = plt.subplots(figsize=(7, 3.4)); ax.imshow(heat.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels([f"{d:.0%}" for d in heat.columns]); ax.set_yticks(range(len(heat.index))); ax.set_yticklabels(heat.index, fontsize=8)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]): ax.text(j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_xlabel("needle depth (0% = far back)"); ax.set_title(f"Needle retrieval, k = 20% of tokens"); plt.tight_layout(); plt.savefig("results/gauntlet_needle.png", dpi=150); plt.show()

In [ ]:
# ---- Optional length-generalisation test (RUN_LONG = True on the server): attention at 2T tokens, policies unchanged ----
if RUN_LONG:
    T2 = 2 * T; LONG_B = [0.1, 0.2, 0.3]; LDATA, LKIND = {}, {}
    for nm in BIG:
        ids, free = FREE[nm]; ids_full = tok(TEXTS[nm], return_tensors="pt", add_special_tokens=False).input_ids[0]
        starts = [p for p in free if p + 1 in free][:2] or free[:2]
        for i, p in enumerate(starts):
            seq = ids_full[p * T: p * T + T2]
            if len(seq) < T2: continue
            LDATA[("long: " + nm, i)] = attention_matrix(model, seq)[0]; LKIND["long: " + nm] = "long"
    DATA.update(LDATA); KIND.update(LKIND); add_optima(list(LDATA), LONG_B)
    sub = {k: DATA[k] for k in LDATA}
    df_long = run_ratio(sub, OPT, OPTM, KIND, SPECS, LONG_B, TRAIN_BUDGETS, pool=(DATA, OPT, KIND))
    df_long.to_csv("results/gauntlet_long.csv", index=False)
    print(f"=== trained at T={T}, tested at T={T2} ==="); display(leaderboard(df_long)[cols].round(3))
else:
    print("RUN_LONG = False: skipped.")

In [ ]:
# ---- Test 6: Chameleon ablations on the same battery + regret check + what it learned ----
ABL = chameleon_ablations()
df_abl = run_ratio(DATA, OPT, OPTM, KIND, ABL, TEST_BUDGETS, TRAIN_BUDGETS); df_abl.to_csv("results/gauntlet_ablation.csv", index=False)
ta = df_abl[df_abl.split == "test"]
print("Ablations (TEST sequences, pure + shift). Each row removes or changes ONE thing:")
display(leaderboard(ta)[["mean ratio", "CI lo", "CI hi", "worst text", "mean rank"]].round(3))
print("\nPaired difference vs full Chameleon (negative = the variant is WORSE, so that component helps):")
dd = vs_leader(ta, "Chameleon"); dd["variant_minus_chameleon"] = -dd["leader_minus_policy"]; display(dd[["policy", "variant_minus_chameleon", "p_holm"]].round(4))

rc = regret_check({k: A for k, A in DATA.items() if split_of(k[1]) == "test" and KIND[k[0]] != "long"}, [0.1, 0.2, 0.3])
print(f"\nRegret bound holds in {rc.holds.mean():.0%} of {len(rc)} runs; median regret / bound = {(rc.regret / rc.bound).median():.2f}")
print("Best fixed expert in hindsight, by text:"); display(rc.assign(expert=rc.best_fixed_expert.map(dict(enumerate(EXPERTS)))).groupby("text")["expert"].agg(lambda s: s.value_counts().to_dict()))

# what did it learn? mean weights over time, one test sequence per text (dotted lines = where the text type changes)
texts = list(KIND); texts = [t for t in texts if KIND[t] != "long"]
fig, axes = plt.subplots(1, len(texts), figsize=(4.2 * len(texts), 3.4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), texts):
    key = [k for k in DATA if k[0] == name and split_of(k[1]) == "test"][0]; pol = Chameleon(); A = DATA[key]; Tn = A.shape[0]
    simulate(A, int(0.15 * Tn), pol); Wt = np.array(pol.w_hist)
    ax.stackplot(range(len(Wt)), Wt.T, labels=[EXPERTS[i] for i in pol.sel]); ax.set_title(name, fontsize=9); ax.set_xlabel("step")
    if KIND[name] == "shift":
        for b in (SEG, 2 * SEG): ax.axvline(b, color="k", ls=":", lw=1.2)
np.atleast_1d(axes)[0].set_ylabel("Chameleon's weight on each expert"); np.atleast_1d(axes)[-1].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.savefig("results/chameleon_weights.png", dpi=150); plt.show()

In [ ]:
# ---- Test 7: overhead ----
ov = overhead(SPECS, T=T, budget=0.2)
lrn = test.groupby("policy")["us_per_step"].mean().reindex([s.name for s in SPECS if s.learned]).round(1)
display(ov); print("learned policies (measured inside the scoreboard, us/step):", lrn.to_dict())

In [ ]:
# ---- numbers to quote (decided by the pre-registered rule, not by us) ----
print("=== 10e: Gauntlet verdict ===")
print(verdict(df_all, df_adv))
print()
lb = leaderboard(test[test.kind == "pure"]); print("Ranking by mean ratio (pure, TEST):", ", ".join(f"{p} {r:.3f}" for p, r in lb["mean ratio"].items()))
if (test.kind == "shift").any():
    ls = leaderboard(test[test.kind == "shift"]); print("Ranking under distribution shift:    ", ", ".join(f"{p} {r:.3f}" for p, r in ls["mean ratio"].items()))
print("Adversarial worst ratio found:       ", ", ".join(f"{r.policy} {r.worst_ratio:.2f}" for r in df_adv.sort_values("worst_ratio", ascending=False).itertuples()))
print("dPPL % (lower better):               ", ", ".join(f"{p} {v:.1f}" for p, v in pv.mean().sort_values().items()))
print(f"Proxy validity (Kendall tau):         {tau:+.2f}")
print("Needle accuracy:                     ", ", ".join(f"{p} {r.acc:.0%}" for p, r in acc.sort_values('acc', ascending=False).iterrows()))
print(f"Chameleon regret bound held in {rc.holds.mean():.0%} of runs.")
print("\nIf Chameleon does not win, say so. A hybrid that ties the best simple rule but is more robust, or an ensemble that adapts but pays a price, is still a finding.")

## 11. Automatic summary of findings (use these numbers in your report)

In [ ]:
on = df[df.online]
best = on.groupby(["text", "policy"])["ratio"].mean().unstack().idxmax(1)
print("Best ONLINE policy by competitive ratio:");  [print(f"  {t:10s}: {p}") for t, p in best.items()]
print("\nMean competitive ratio per policy (all texts, budgets):")
print(df.groupby("policy")["ratio"].mean().reindex(POLICY_ORDER).round(3).to_string())
g = df[df.policy == "Belady-Greedy (offline)"]["ratio"]
print(f"\nBelady-greedy reaches {g.mean():.4f} of optimum on average (worst {g.min():.4f})"
      + (" -> greedy is NOT optimal when eviction is permanent" if g.min() < 0.9999 else ""))
print(f"Spearman correlation between attention retained and ΔPPL: {rho:.2f}")
best_ppl = df2.loc[df2.groupby(['text', 'budget'])['dppl_pct'].idxmin(), ['text', 'budget', 'policy', 'dppl_pct']]
print("\nLowest perplexity loss per setting (is the attention-optimal schedule also the perplexity-best?):")
print(best_ppl.to_string(index=False))
print("\nNeedle accuracy:"); print(acc.round(2).to_string())

In [ ]:
import shutil; shutil.make_archive("kv_eviction_results", "zip", "results")
try:
    from google.colab import files; files.download("kv_eviction_results.zip")
except Exception:
    print("Saved kv_eviction_results.zip")

## Notes & limitations (say these in your viva — it shows maturity)
- One eviction decision is shared across all layers and heads (real H2O decides per head). This keeps the problem a clean single online problem.
- The "optimum" is optimal for **attention retained**, a proxy. Experiment 2 checks how well that proxy tracks real quality.
- Attention after eviction can differ from full-cache attention; the simulation is an approximation, while Experiments 2 and 3 are exact.
- Small model (0.5B) and short sequences (512) to fit a free GPU; the min-cost flow has O(T²) arcs, so the exact optimum is practical only for a few thousand tokens.
- **Chameleon / Gauntlet (10e):** the regret guarantee covers the expert *weights* on a surrogate loss (ranking quality on observed attention), not the competitive ratio; the rank-averaged decision is a heuristic. Chameleon costs roughly 10-60x more python time per decision than the O(1)/O(k) rules (five argsorts per step; a vectorised or per-head implementation would cut this). Hyper-parameters were fixed from theory (eta = sqrt(8 ln N / T)) and a synthetic dev set, never tuned on the real TEST sequences.
